# Native private-feedback teleportation

Preserve the original plus input and Bob-only output. The frozen native qfunc
uses explicitly deferred private measurements: coherent Pauli corrections retain
Alice's outcome qubits until tracing them out. This preserves Bob's output
channel, including complex coherence. It is not a generic mid-circuit measurement,
reset operation or physical classical communication demonstration.


## Express the protocol before reading its statistics

The complete native qfunc below prepares the message and entangled pair, then
composes the transfer and correction operations. This implementation uses deferred
measurement: coherent controls replace intermediate measurement and classical
feed-forward. It is a local unitary simulation of the protocol, not a distributed
communication service or a claim that no classical communication is needed in
physical teleportation.

The user specifies the protocol and quantum rotations. FlagQuantum validates the
composition, builds the circuit and executes it; no physical wire indices or
manual output decoder appear in the qfunc. `drop` hides the Alice registers from
returned ports without claiming they are clean ancillas. The Bob marginal and
density matrix distinguish those retained environment bits from compiler work.

Try theta=0.73 and phi=-0.41 in the visible binding. Bob's density matrix should
match the prepared complex message state. Measurement probabilities alone would
miss its relative phase, which the independent tests check.


In [ ]:
import math
from math import pi
from flagquantum import synth
from flagquantum.synth import (
    CX,
    H,
    Output,
    QBit,
    R,
    allocate,
    control,
    drop,
    phase,
    qfunc,
)


@qfunc
def send(alice: QBit, bell: QBit, bob: QBit) -> None:
    CX(alice, bell)
    H(alice)
    CX(bell, bob)
    control(alice, lambda: phase(bob, pi))


@qfunc
def teleport(theta: float, phi: float, bob_qubit: Output[QBit]) -> None:
    alice: QBit = QBit()
    bell: QBit = QBit()
    allocate(bob_qubit)
    allocate(alice)
    allocate(bell)
    R(theta, pi / 2, alice)
    phase(alice, phi)
    H(bell)
    CX(bell, bob_qubit)
    send(alice, bell, bob_qubit)
    drop(alice)
    drop(bell)


import flagquantum as fq
from examples.synthesis.algorithms.foundational.quantum_teleportation.main import (
    build_program,
)
from examples.synthesis.algorithms.foundational.quantum_teleportation.overview.workflow import (
    run_workflow,
)
from flagquantum.drawer import draw

program = synth.synthesize(
    teleport,
    classical_inputs={"theta": pi / 2, "phi": 0.0},
    max_qubits=4,
    max_workspace=1,
    memory_limit_bytes=1024**2,
)
print("Actual resources:", program.report["resources"])
print("Clean temporary workspace:", program.report["workspace_qubits"])
print(draw(program.circuit, show_all_qubits=True))


## Bob's native named readout

The modeled output is Bob's one-bit register. Alice's retained outcome qubits are
an environment; they are not reset or reported as clean workspace. The workflow
handles physical state axes internally.


In [ ]:
result = fq.run(program)
print(dict(program.probabilities(result, registers=("bob_qubit",))))


## Explicit bounded local observations

The source `sample` call has no explicit shot count. Here choose 1000 local
observations and seed 3 explicitly; do not claim the same SDK default or remote
sample sequence. Report actual samples and Bob's reduced density matrix.


In [ ]:
workflow = run_workflow(shots=1000, seed=3)
print("Actual Bob observations:", dict(workflow.observations))
print("Actual shot count:", sum(count for bit, count in workflow.observations))
print("Bob density matrix:", workflow.density)


## A nonzero complex input

Also prepare cos(theta/2)|0> + exp(i*phi)sin(theta/2)|1>. This checks relative
complex phase through the native protocol, beyond computational-basis counts.
Independent acceptance explicitly enumerates the four actual measurement
branches and checks their output density, full purification and clean workspace.


In [ ]:
complex_workflow = run_workflow(theta=0.73, phi=-0.41, shots=1000, seed=3)
print("Complex-input Bob observations:", dict(complex_workflow.observations))
print("Complex-input Bob density:", complex_workflow.density)
print("Complex-input resources:", complex_workflow.program.report["resources"])
